<a href="https://colab.research.google.com/github/camille2019/Women-Capital-Trial-Analysis/blob/main/propensity_matching_psm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [129]:
!pip install psmpy


In [130]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

from psmpy import PsmPy
from psmpy.functions import cohenD
from psmpy.plotting import *

In [131]:
from psmpy import PsmPy


In [132]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [133]:
save_path = '/content/drive/MyDrive/Capital_trials/'
# full_meta = pd.read_csv(save_path + 'full_crime_meta_.csv')
# ohe_meta = pd.read_csv('/content/drive/MyDrive/Capital_trials/ohe_metadata_only.csv')

ohe_meta = pd.read_csv('/content/ohe_meta_all_categories.csv')

In [134]:
ohe_meta

ohe_meta = ohe_meta.drop(['Unnamed: 0', 'Unnamed: 0.1'], axis=1)


In [135]:
ohe_meta.columns

Index(['Name', 'crime_cont_threat', 'crime_disregard_human_life',
       'crime_manner_mayhem', 'crime_retaliation', 'crime_old_disabled_victim',
       'crime_weapon_arson', 'crime_weapon', 'crime_aggrevated_assult',
       'crime_attempted_rape', 'crime_child_victim', 'crime_public_risk',
       'crime_premeditated', 'crime_multiple_felonies',
       'crime_especially_cruel', 'crime_gang_murder', 'crime_bodily_injury',
       'crime_interfere_arrest', 'crime_kidnapping', 'crime_larceny',
       'crime_lewd_acts_child', 'crime_lying_in_wait', 'crime_police_victim',
       'crime_handgun', 'crime_weapon_poison', 'crime_prior_conv',
       'crime_prior_off', 'crime_rape', 'crime_sexual_assult',
       'crime_shooting_car', 'crime_sodomy', 'crime_torture',
       'crime_unlawful_firearm', 'victim_list', 'victim_child',
       'victim_child_care', 'victim_partner', 'victim_ex_partner',
       'victim_stranger', 'victim_family', 'victim_familiar', 'State_AL',
       'State_AZ', 'State_CA',

In [136]:
ohe_meta.loc[ohe_meta['crime_manner_mayhem'] == True, ['Name', 'gender_woman']]


,Name,gender_woman
9,"Matthews, Damon Roshun",False
18,"Johnson, Dexter Darnell",False
113,Belinda Magaña,True
154,Veronica Gonzales,True


In [137]:
race_cols =['Race_Asian',
       'Race_Black', 'Race_Latinx', 'Race_Native American', 'Race_White',]

In [138]:
state_cols = ['State_AL',
       'State_AZ', 'State_CA', 'State_FL', 'State_GA', 'State_ID', 'State_KY',
       'State_LA', 'State_MS', 'State_NC', 'State_OH', 'State_OK', 'State_PA',
       'State_SC', 'State_TN', 'State_TX']

In [139]:
crime_cols = ['crime_cont_threat', 'crime_disregard_human_life',
       'crime_manner_mayhem', 'crime_retaliation', 'crime_old_disabled_victim',
       'crime_weapon_arson', 'crime_weapon', 'crime_aggrevated_assult',
       'crime_attempted_rape', 'crime_child_victim', 'crime_public_risk',
       'crime_premeditated', 'crime_multiple_felonies',
       'crime_especially_cruel', 'crime_gang_murder', 'crime_bodily_injury',
       'crime_interfere_arrest', 'crime_kidnapping', 'crime_larceny',
       'crime_lewd_acts_child', 'crime_lying_in_wait', 'crime_police_victim',
       'crime_handgun', 'crime_weapon_poison', 'crime_prior_conv',
       'crime_prior_off', 'crime_rape', 'crime_sexual_assult',
       'crime_shooting_car', 'crime_sodomy', 'crime_torture',
       'crime_unlawful_firearm', ]

In [140]:
victim_relationship_cols = ['victim_child',
       'victim_child_care', 'victim_partner', 'victim_ex_partner',
       'victim_stranger', 'victim_family', 'victim_familiar']

In [ ]:
sentence_band_cols  = [ 'sentence_band_1980-1984', 'sentence_band_1985-1989',
       'sentence_band_1990-1994', 'sentence_band_1995-1999',
       'sentence_band_2000-2004', 'sentence_band_2005-2009',
       'sentence_band_2010-2014', 'sentence_band_2015-2019',
       'sentence_band_2020-2024', 'sentence_band_2070-2074']

In [ ]:
parent_col = ['parent_binary']

In [ ]:
state_race_crime_cols = state_cols + race_cols + crime_cols

In [ ]:
race_crime_cols = race_cols + crime_cols

In [ ]:
def add_propensity(propensity_columns, treatment_column, df_encoded, df_meta, matched_group_name, add_score_col= True, cal = None, rep=False,  h_m = 1, m= None):

  psm_df = df_encoded[propensity_columns + [treatment_column]].copy()


  psm_df = psm_df.reset_index(drop=True)
  psm_df["row_id"] = psm_df.index

  psm = PsmPy(psm_df, treatment=treatment_column, indx="row_id", exclude=[])

  psm.logistic_ps(balance=True)

  if h_m > 1:
    if m is None:
      psm.kdtree_matched_12n(matcher='propensity_logit', how_many = h_m )
    else:
      psm.kdtree_matched_12n(matcher= m, how_many = h_m )


  else:
    if m is None:
      psm.kdtree_matched(matcher='propensity_logit', caliper=cal, replacement = rep )
    else:
      psm.kdtree_matched(matcher=m, caliper=cal, replacement = rep )


  pred = psm.predicted_data

  df_meta = df_meta.reset_index(drop=True)

  scores_col_name = matched_group_name + '_prop_scores'

  if add_score_col:

    score_map = pred[["row_id", "propensity_score"]].drop_duplicates("row_id")
    df_meta = df_meta.merge(score_map, left_index=True, right_on="row_id", how="left")
    df_meta = df_meta.drop(columns=["row_id"])
    df_meta = df_meta.rename(columns={"propensity_score": matched_group_name + "_prop_scores"})

  # psm.kdtree_matched(matcher="propensity_score", replacement=False, caliper=None)
  # matched_df = psm.df_matched
  # matched_ids = set(psm.df_matched["row_id"].unique())
  # df_meta = df_meta.reset_index(drop=True)

  # binary_col_name = 'psm_' + matched_group_name

  # df_meta[binary_col_name] = df_meta.index.isin(matched_ids)
  matched_ids = set(psm.df_matched["row_id"].unique())
  df_meta[ matched_group_name] = df_meta.index.isin(matched_ids)

  return df_meta



In [ ]:
def find_caliper(updated_df, propensity_score_col):
  updated_df['logit_ps'] = np.log(updated_df[propensity_score_col] / (1 - updated_df[propensity_score_col]))

  std_logit = updated_df['logit_ps'].std()

  caliper = 0.2 * std_logit

  tight_caliper = 0.1 * std_logit

  loose_caliper = 0.5 * std_logit

  print(f"Recommended Caliper (.2 * std): {caliper}")
  print(f"Tight Caliper (.1 * std): {tight_caliper}")
  print(f"Loose Caliper (.5 * std): {loose_caliper}")

In [ ]:
def check_calipers_same_match(updated_df, match_version1, match_version2):
  version1 = set(updated_df.loc[updated_df[match_version1] == True, 'def_name'].to_list() )
  version2 = set(updated_df.loc[updated_df[match_version2] == True, 'def_name'].to_list() )
  groups_equal = version1 == version2
  return groups_equal

In [ ]:
df = pd.DataFrame()
df['def_name'] = ohe_meta['Name']
df['def_woman'] = ohe_meta['gender_woman']

# Propensity matching for Crime alone (RQ1)


In [ ]:
ohe_meta.columns

Index(['Name', 'crime_cont_threat', 'crime_disregard_human_life',
       'crime_manner_mayhem', 'crime_retaliation', 'crime_old_disabled_victim',
       'crime_weapon_arson', 'crime_weapon', 'crime_aggrevated_assult',
       'crime_attempted_rape', 'crime_child_victim', 'crime_public_risk',
       'crime_premeditated', 'crime_multiple_felonies',
       'crime_especially_cruel', 'crime_gang_murder', 'crime_bodily_injury',
       'crime_interfere_arrest', 'crime_kidnapping', 'crime_larceny',
       'crime_lewd_acts_child', 'crime_lying_in_wait', 'crime_police_victim',
       'crime_handgun', 'crime_weapon_poison', 'crime_prior_conv',
       'crime_prior_off', 'crime_rape', 'crime_sexual_assult',
       'crime_shooting_car', 'crime_sodomy', 'crime_torture',
       'crime_unlawful_firearm', 'victim_list', 'victim_child',
       'victim_child_care', 'victim_partner', 'victim_ex_partner',
       'victim_stranger', 'victim_family', 'victim_familiar', 'State_AL',
       'State_AZ', 'State_CA',

In [142]:
crime_only_prop = add_propensity(crime_cols, 'gender_woman', ohe_meta, df, 'crime_only_match')

In [143]:
crime_only_prop

,def_name,def_woman,crime_only_match_prop_scores,psm_crime_only_match
47,"Gaxiola, Albert Robert",False,0.539457,True
48,"Reid, Albert Ezron",False,0.517266,False
49,"Lackey, Andrew",False,0.494814,False
50,"Milam, Blaine Keith",False,0.506324,False
51,"Kennedy, Carlos",False,0.483225,False
...,...,...,...,...
42,Tina Brown,True,0.781554,False
43,Valerie Martin,True,0.639901,True
44,Veronica Gonzales,True,0.595728,False
45,Virginia Caudill,True,0.499680,True


In [144]:
find_caliper(crime_only_prop, 'crime_only_match_prop_scores')

Recommended Caliper (.2 * std): 0.10177805244148999
Tight Caliper (.1 * std): 0.050889026220744996
Loose Caliper (.5 * std): 0.25444513110372496


In [145]:
crime_only_prop = add_propensity(crime_cols, 'gender_woman',  ohe_meta, crime_only_prop, 'crime_only_match_cal_rec', cal=0.10177805244148999, add_score_col=False , m ='propensity_score')
crime_only_prop = add_propensity(crime_cols, 'gender_woman', ohe_meta, crime_only_prop, 'crime_only_match_cal_tight', cal=0.050889026220744996, add_score_col=False,  m ='propensity_score')
crime_only_prop = add_propensity(crime_cols, 'gender_woman', ohe_meta, crime_only_prop, 'crime_only_match_cal_loose', cal=0.25444513110372496, add_score_col=False,  m ='propensity_score')

crime_only_prop = add_propensity(crime_cols, 'gender_woman', ohe_meta, crime_only_prop, 'crime_only_match_cal_tight_replacement', cal=0.050889026220744996, add_score_col=False, rep = True,  m ='propensity_score')
crime_only_prop = add_propensity(crime_cols, 'gender_woman', ohe_meta, crime_only_prop, 'crime_only_match_cal_loose_replacement', cal=0.25444513110372496, add_score_col=False, rep= True,  m ='propensity_score')


/usr/local/lib/python3.12/dist-packages/psmpy/psmpy.py:374: UserWarning: Some values do not have a match. These are dropped for purposes of establishing a matched dataframe, and subsequent calculations and plots (effect size). If you do not wish this to be the case please set drop_unmatched=False
  warnings.warn('Some values do not have a match. These are dropped for purposes of establishing a matched dataframe, and subsequent calculations and plots (effect size). If you do not wish this to be the case please set drop_unmatched=False')
/usr/local/lib/python3.12/dist-packages/psmpy/psmpy.py:374: UserWarning: Some values do not have a match. These are dropped for purposes of establishing a matched dataframe, and subsequent calculations and plots (effect size). If you do not wish this to be the case please set drop_unmatched=False
  warnings.warn('Some values do not have a match. These are dropped for purposes of establishing a matched dataframe, and subsequent calculations and plots (eff

In [146]:
crime_only_prop

,def_name,def_woman,crime_only_match_prop_scores,psm_crime_only_match,logit_ps,psm_crime_only_match_cal_rec,psm_crime_only_match_cal_tight,psm_crime_only_match_cal_loose,psm_crime_only_match_cal_tight_replacement,psm_crime_only_match_cal_loose_replacement
0,"Gaxiola, Albert Robert",False,0.539457,True,0.158157,True,True,True,False,False
1,"Reid, Albert Ezron",False,0.517266,False,0.069090,True,False,True,False,False
2,"Lackey, Andrew",False,0.494814,False,-0.020746,False,False,True,False,False
3,"Milam, Blaine Keith",False,0.506324,False,0.025296,False,False,True,False,False
4,"Kennedy, Carlos",False,0.483225,False,-0.067127,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
152,Tina Brown,True,0.781554,False,1.274746,False,False,False,True,True
153,Valerie Martin,True,0.639901,True,0.574932,False,False,True,True,True
154,Veronica Gonzales,True,0.595728,False,0.387697,True,False,True,True,True
155,Virginia Caudill,True,0.499680,True,-0.001280,True,True,True,True,True


In [147]:
crime_only_prop[crime_only_prop['psm_crime_only_match_cal_rec']==True]['def_woman'].value_counts()

,count
def_woman,
False,33
True,33


In [ ]:
are_equal = crime_only_prop['psm_crime_only_match_cal_rec'].equals(crime_only_prop['psm_crime_only_match_cal_tight_replacement'])
are_equal

False

In [ ]:
crime_only_prop.to_csv('crime_matched_group_options.csv')

#Propensity matching for state + race+ crime

In [ ]:
state_race_crime_props = add_propensity(state_race_crime_cols, 'gender_woman', ohe_meta, df, 'state_race_crime')


In [ ]:
state_race_crime_props

,def_name,def_woman,state_race_crime_prop_scores,psm_state_race_crime
47,"Gaxiola, Albert Robert",False,0.649177,True
48,"Reid, Albert Ezron",False,0.437303,False
49,"Lackey, Andrew",False,0.604984,False
50,"Milam, Blaine Keith",False,0.473524,True
51,"Kennedy, Carlos",False,0.575836,False
...,...,...,...,...
42,Tina Brown,True,0.857919,True
43,Valerie Martin,True,0.664623,True
44,Veronica Gonzales,True,0.584787,True
45,Virginia Caudill,True,0.706206,True


In [ ]:
find_caliper(state_race_crime_props, 'state_race_crime_prop_scores')


Recommended Caliper (.2 * std): 0.14927040297477284
Tight Caliper (.1 * std): 0.07463520148738642
Loose Caliper (.5 * std): 0.3731760074369321


In [ ]:

state_race_crime_props = add_propensity(crime_cols, 'gender_woman',  ohe_meta, state_race_crime_props, 'state_race_crime_match_cal_rec', cal=0.14927040297477284, add_score_col=False , m ='propensity_score')
state_race_crime_props = add_propensity(crime_cols, 'gender_woman', ohe_meta, state_race_crime_props, 'state_race_crime_match_cal_tight', cal=0.07463520148738642, add_score_col=False,  m ='propensity_score')
state_race_crime_props = add_propensity(crime_cols, 'gender_woman', ohe_meta, state_race_crime_props, 'state_race_crime_match_cal_loose', cal=0.3731760074369321, add_score_col=False,  m ='propensity_score')

state_race_crime_props = add_propensity(crime_cols, 'gender_woman', ohe_meta, state_race_crime_props, 'state_race_crime_match_cal_rec_replacement', cal=0.07463520148738642, add_score_col=False, rep = True,  m ='propensity_score')
state_race_crime_props = add_propensity(crime_cols, 'gender_woman', ohe_meta, state_race_crime_props, 'state_race_crime_match_cal_loose_replacement', cal=0.3731760074369321, add_score_col=False, rep= True,  m ='propensity_score')
state_race_crime_props = add_propensity(crime_cols, 'gender_woman', ohe_meta, state_race_crime_props, 'state_race_crime_match_cal_loose_replacement', cal=0.14927040297477284, add_score_col=False, rep = True,  m ='propensity_score')

state_race_crime_props = add_propensity(crime_cols, 'gender_woman', ohe_meta, state_race_crime_props, 'state_race_crime_match_cal_tight_replacement', cal=0.07463520148738642, add_score_col=False, rep = True,  m ='propensity_score')
state_race_crime_props = add_propensity(crime_cols, 'gender_woman', ohe_meta, state_race_crime_props, 'state_race_crime_match_cal_loose_replacement', cal=0.3731760074369321, add_score_col=False, rep= True,  m ='propensity_score')
state_race_crime_props = add_propensity(crime_cols, 'gender_woman', ohe_meta, state_race_crime_props, 'state_race_crime_match_cal_rec_replacement', cal=0.14927040297477284, add_score_col=False, rep = True,  m ='propensity_score')


/usr/local/lib/python3.12/dist-packages/psmpy/psmpy.py:374: UserWarning: Some values do not have a match. These are dropped for purposes of establishing a matched dataframe, and subsequent calculations and plots (effect size). If you do not wish this to be the case please set drop_unmatched=False
  warnings.warn('Some values do not have a match. These are dropped for purposes of establishing a matched dataframe, and subsequent calculations and plots (effect size). If you do not wish this to be the case please set drop_unmatched=False')
/usr/local/lib/python3.12/dist-packages/psmpy/psmpy.py:374: UserWarning: Some values do not have a match. These are dropped for purposes of establishing a matched dataframe, and subsequent calculations and plots (effect size). If you do not wish this to be the case please set drop_unmatched=False
  warnings.warn('Some values do not have a match. These are dropped for purposes of establishing a matched dataframe, and subsequent calculations and plots (eff

In [ ]:
state_race_crime_props = add_propensity(crime_cols, 'gender_woman', ohe_meta, state_race_crime_props, 'state_race_crime_match_cal_tight_hm2', cal=0.07463520148738642, add_score_col=False, h_m=2 ,m ='propensity_score')
state_race_crime_props = add_propensity(crime_cols, 'gender_woman', ohe_meta, state_race_crime_props, 'state_race_crime_match_cal_loose_hm2', cal=0.3731760074369321, add_score_col=False,  h_m=2 ,  m ='propensity_score')
state_race_crime_props = add_propensity(crime_cols, 'gender_woman', ohe_meta, state_race_crime_props, 'state_race_crime_match_cal_rec_hm2', cal=0.14927040297477284, add_score_col=False,  h_m=2 ,  m ='propensity_score')

In [ ]:
state_race_crime_props

,def_name,def_woman,state_race_crime_prop_scores,psm_state_race_crime,logit_ps,psm_state_race_crime_match_cal_rec,psm_state_race_crime_match_cal_tight,psm_state_race_crime_match_cal_loose,psm_state_race_crime_match_cal_rec_replacement,psm_state_race_crime_match_cal_loose_replacement,psm_state_race_crime_match_cal_tight_replacement,psm_state_race_crime_match_cal_tight_hm2,psm_state_race_crime_match_cal_loose_hm2,psm_state_race_crime_match_cal_rec_hm2
0,"Gaxiola, Albert Robert",False,0.649177,True,0.615423,True,True,True,False,False,False,True,True,True
1,"Reid, Albert Ezron",False,0.437303,False,-0.252115,True,False,True,False,False,False,True,True,True
2,"Lackey, Andrew",False,0.604984,False,0.426274,False,False,True,False,False,False,True,True,True
3,"Milam, Blaine Keith",False,0.473524,True,-0.106002,True,False,True,False,False,False,True,True,True
4,"Kennedy, Carlos",False,0.575836,False,0.305703,False,False,False,False,False,False,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152,Tina Brown,True,0.857919,True,1.798116,False,False,True,True,True,True,True,True,True
153,Valerie Martin,True,0.664623,True,0.683965,True,False,True,True,True,True,True,True,True
154,Veronica Gonzales,True,0.584787,True,0.342455,True,False,True,True,True,True,True,True,True
155,Virginia Caudill,True,0.706206,True,0.877029,True,True,True,True,True,True,True,True,True


In [ ]:
#check_calipers_same_match(updated_df, match_version1, match_version2)#
check_calipers_same_match(state_race_crime_props,'psm_state_race_crime_match_cal_rec','psm_state_race_crime_match_cal_loose' )

False

In [ ]:
state_race_crime_props.to_csv('state_race_crime_match_options.csv')

#Match for Parents (RQ2)

In [ ]:
def_is_parent = ohe_meta[(ohe_meta['parent_binary']==True) ]

In [148]:

def_is_parent

,Name,crime_cont_threat,crime_disregard_human_life,crime_manner_mayhem,crime_retaliation,crime_old_disabled_victim,crime_weapon_arson,crime_weapon,crime_aggrevated_assult,crime_attempted_rape,...,sentence_band_2020-2024,sentence_band_2070-2074,Race_Asian,Race_Black,Race_Latinx,Race_Native American,Race_White,gender_man,gender_woman,parent_binary
1,"Reid, Albert Ezron",0,0,0,0,0,0,0,0,0,...,False,False,False,True,False,False,False,True,False,1
3,"Milam, Blaine Keith",0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,True,True,False,1
5,"Ricks, Cedric Allen",0,0,0,0,0,0,0,0,0,...,False,False,False,True,False,False,False,True,False,1
7,"Sepulvado, Christopher",1,1,0,0,0,1,0,0,0,...,False,False,False,False,False,False,True,True,False,1
8,"Ray, Clarence Jr",0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,True,False,True,False,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152,Tina Brown,0,0,0,0,0,0,0,0,0,...,False,False,False,True,False,False,False,False,True,1
153,Valerie Martin,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,True,False,True,1
154,Veronica Gonzales,0,0,1,0,0,0,0,0,0,...,False,False,False,False,True,False,False,False,True,1
155,Virginia Caudill,0,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,True,False,True,1


In [ ]:
def_is_parent['gender_woman'].value_counts()

,count
gender_woman,
False,46
True,41


#Match for Victim Relationship (partner or child) (RQ3)

In [ ]:
victim_relationship_cols = ['victim_child',
       'victim_child_care', 'victim_partner', 'victim_ex_partner',
       'victim_stranger', 'victim_family', 'victim_familiar']


victim_partner_or_child = ohe_meta[(ohe_meta['victim_partner']==True) | (ohe_meta['victim_child']==True) | (ohe_meta['victim_ex_partner']==True)  ]

In [ ]:
victim_partner_or_child[['Name', 'gender_woman']]

,Name,gender_woman
1,"Reid, Albert Ezron",False
5,"Ricks, Cedric Allen",False
7,"Sepulvado, Christopher",False
20,"Hernandez, Fabian",False
27,"Garton, Todd Jesse",False
38,"Brooks, Donald Lewis Jr",False
42,"Calvert, James",False
51,"Rubio, John Allen (2003)",False
52,"Rubio, John Allen (2007)",False
54,"DeBlase, John Joseph",False


In [150]:
victim_partner= ohe_meta[(ohe_meta['victim_partner']==True) | (ohe_meta['victim_ex_partner']==True)]
victim_partner[['Name', 'gender_woman']]

,Name,gender_woman
5,"Ricks, Cedric Allen",False
20,"Hernandez, Fabian",False
27,"Garton, Todd Jesse",False
38,"Brooks, Donald Lewis Jr",False
42,"Calvert, James",False
97,"San Nicolas, Rodney Jesse",False
111,Angelina Rodriguez,True
114,Blanche Taylor Moore,True
115,Brenda E. Andrew,True
124,Donna Marie Roberts,True


In [ ]:
victim_child= ohe_meta[(ohe_meta['victim_child']==True)]
victim_child[['Name', 'gender_woman']]

,Name,gender_woman
1,"Reid, Albert Ezron",False
7,"Sepulvado, Christopher",False
51,"Rubio, John Allen (2003)",False
52,"Rubio, John Allen (2007)",False
54,"DeBlase, John Joseph",False
67,"Hunter, Lamont",False
68,"Atkins, Randy Lynn",False
76,"Northcutt, Robert Clinton",False
88,"Holiday, Raphael D.",False
95,"Roberson, Robert Leslie",False


# Match for Race (RQ4)

In [ ]:
race_only_props = add_propensity(race_cols, 'gender_woman', ohe_meta, df, 'race_only')
race_only_props

,def_name,def_woman,race_only_prop_scores,psm_race_only
47,"Gaxiola, Albert Robert",False,0.501999,False
48,"Reid, Albert Ezron",False,0.394231,False
49,"Lackey, Andrew",False,0.546026,False
50,"Milam, Blaine Keith",False,0.546026,True
51,"Kennedy, Carlos",False,0.394231,False
...,...,...,...,...
42,Tina Brown,True,0.436736,True
43,Valerie Martin,True,0.573249,True
44,Veronica Gonzales,True,0.416497,True
45,Virginia Caudill,True,0.559937,True


In [ ]:
find_caliper(race_only_props,'race_only_prop_scores' )

Recommended Caliper (.2 * std): 0.07477559563381155
Tight Caliper (.1 * std): 0.03738779781690577
Loose Caliper (.5 * std): 0.18693898908452886


In [ ]:
race_only_props = add_propensity(race_cols, 'gender_woman', ohe_meta, race_only_props, 'race_only_cal_rec', cal=0.07477559563381155 )
race_only_props = add_propensity(race_cols, 'gender_woman', ohe_meta, race_only_props, 'race_only_cal_loose', cal = 0.18693898908452886)
race_only_props = add_propensity(race_cols, 'gender_woman', ohe_meta, race_only_props, 'race_only_cal_tight', cal =0.03738779781690577 )


/usr/local/lib/python3.12/dist-packages/psmpy/psmpy.py:374: UserWarning: Some values do not have a match. These are dropped for purposes of establishing a matched dataframe, and subsequent calculations and plots (effect size). If you do not wish this to be the case please set drop_unmatched=False
  warnings.warn('Some values do not have a match. These are dropped for purposes of establishing a matched dataframe, and subsequent calculations and plots (effect size). If you do not wish this to be the case please set drop_unmatched=False')
/usr/local/lib/python3.12/dist-packages/psmpy/psmpy.py:374: UserWarning: Some values do not have a match. These are dropped for purposes of establishing a matched dataframe, and subsequent calculations and plots (effect size). If you do not wish this to be the case please set drop_unmatched=False
  warnings.warn('Some values do not have a match. These are dropped for purposes of establishing a matched dataframe, and subsequent calculations and plots (eff

In [ ]:
race_only_props

,def_name,def_woman,race_only_prop_scores,psm_race_only,logit_ps,race_only_cal_rec_prop_scores,psm_race_only_cal_rec,race_only_cal_loose_prop_scores,psm_race_only_cal_loose,race_only_cal_tight_prop_scores,psm_race_only_cal_tight
47,"Gaxiola, Albert Robert",False,0.501999,False,0.007998,0.501999,False,0.501999,False,0.501999,False
48,"Reid, Albert Ezron",False,0.394231,False,-0.429561,0.394231,False,0.394231,False,0.394231,False
49,"Lackey, Andrew",False,0.546026,False,0.184629,0.546026,False,0.546026,False,0.546026,False
50,"Milam, Blaine Keith",False,0.546026,True,0.184629,0.546026,False,0.546026,True,0.546026,False
51,"Kennedy, Carlos",False,0.394231,False,-0.429561,0.394231,False,0.394231,False,0.394231,False
...,...,...,...,...,...,...,...,...,...,...,...
42,Tina Brown,True,0.436736,True,-0.254418,0.436736,False,0.436736,True,0.436736,False
43,Valerie Martin,True,0.573249,True,0.295119,0.573249,False,0.573249,True,0.573249,False
44,Veronica Gonzales,True,0.416497,True,-0.337172,0.416497,False,0.416497,True,0.416497,False
45,Virginia Caudill,True,0.559937,True,0.240907,0.559937,True,0.559937,True,0.559937,True


In [ ]:
check_calipers_same_match(race_only_props, 'psm_race_only_cal_rec','psm_race_only' )

False

In [ ]:
race_only_props.to_csv('race_match_options.csv')

#Match for all conditions above (RQ5)

conditions include: crime, victim= partner(ex-partner), victim=child, race

In [ ]:
rq5_cols = crime_cols + victim_relationship_cols + race_cols

In [ ]:
rq5prop = add_propensity(rq5_cols, 'gender_woman', ohe_meta, df, 'rq5_match')

In [ ]:
find_caliper(rq5prop,'rq5_match_prop_scores' )

Recommended Caliper (.2 * std): 0.31784130687687123
Tight Caliper (.1 * std): 0.15892065343843562
Loose Caliper (.5 * std): 0.794603267192178


In [ ]:
rq5prop = add_propensity(race_cols, 'gender_woman', ohe_meta, rq5prop, 'rq5_cal_rec', cal=0.31784130687687123, add_score_col=False )
rq5prop = add_propensity(race_cols, 'gender_woman', ohe_meta, rq5prop, 'rq5_cal_loose', cal = 0.15892065343843562, add_score_col=False)
rq5prop = add_propensity(race_cols, 'gender_woman', ohe_meta, rq5prop, 'rq5_cal_tight', cal =0.794603267192178 , add_score_col=False)


In [ ]:
rq5prop

,def_name,def_woman,rq5_match_prop_scores,psm_rq5_match,logit_ps,psm_rq5_cal_rec,psm_rq5_cal_loose,psm_rq5_cal_tight
0,"Gaxiola, Albert Robert",False,0.166560,True,-1.610209,False,False,False
1,"Reid, Albert Ezron",False,0.194950,False,-1.418163,False,False,False
2,"Lackey, Andrew",False,0.160402,False,-1.655237,True,True,True
3,"Milam, Blaine Keith",False,0.193119,True,-1.429873,True,True,True
4,"Kennedy, Carlos",False,0.187701,False,-1.465015,False,False,False
...,...,...,...,...,...,...,...,...
152,Tina Brown,True,0.794099,True,1.349815,True,True,True
153,Valerie Martin,True,0.957022,True,3.103144,True,True,True
154,Veronica Gonzales,True,0.898552,True,2.181239,True,True,True
155,Virginia Caudill,True,0.903452,False,2.236186,True,True,True


In [ ]:
check_calipers_same_match(rq5prop, 'psm_rq5_match','psm_rq5_cal_rec' )

False

In [ ]:
rec_cal_rq5 = rq5prop[rq5prop['psm_rq5_cal_rec']==True]

In [ ]:
rec_cal_rq5[['def_name', 'def_woman']]

,def_name,def_woman
2,"Lackey, Andrew",False
3,"Milam, Blaine Keith",False
5,"Ricks, Cedric Allen",False
7,"Sepulvado, Christopher",False
8,"Ray, Clarence Jr",False
...,...,...
152,Tina Brown,True
153,Valerie Martin,True
154,Veronica Gonzales,True
155,Virginia Caudill,True


In [ ]:
rec_cal_rq5[ 'def_woman'].value_counts()


,count
def_woman,
False,46
True,46


#Match on rq5 categories  + state and sentencing band

In [ ]:
multi_category = rq5_cols + sentence_band_cols + state_cols

In [ ]:
multi = add_propensity(multi_category, 'gender_woman', ohe_meta, df, 'muti_cat_match')

In [ ]:
multi

,def_name,def_woman,muti_cat_match_prop_scores,psm_muti_cat_match
47,"Gaxiola, Albert Robert",False,0.318248,True
48,"Reid, Albert Ezron",False,0.171211,False
49,"Lackey, Andrew",False,0.151504,False
50,"Milam, Blaine Keith",False,0.242632,False
51,"Kennedy, Carlos",False,0.299126,False
...,...,...,...,...
42,Tina Brown,True,0.889831,True
43,Valerie Martin,True,0.970570,True
44,Veronica Gonzales,True,0.916152,True
45,Virginia Caudill,True,0.925623,False


In [ ]:
find_caliper(multi, 'muti_cat_match_prop_scores')

Recommended Caliper (.2 * std): 0.338364839705966
Tight Caliper (.1 * std): 0.169182419852983
Loose Caliper (.5 * std): 0.845912099264915


In [ ]:
multi = add_propensity(race_cols, 'gender_woman', ohe_meta, multi, 'multi_cal_rec', cal=0.338364839705966, add_score_col=False )
multi = add_propensity(race_cols, 'gender_woman', ohe_meta, multi, 'muti_cal_loose', cal = 0.169182419852983, add_score_col=False)
multi = add_propensity(race_cols, 'gender_woman', ohe_meta, multi, 'muti_cal_tight', cal = 0.845912099264915, add_score_col=False)

In [ ]:
multi

,def_name,def_woman,muti_cat_match_prop_scores,psm_muti_cat_match,logit_ps,psm_multi_cal_rec,psm_muti_cal_loose,psm_muti_cal_tight
0,"Gaxiola, Albert Robert",False,0.318248,True,-0.761834,False,False,False
1,"Reid, Albert Ezron",False,0.171211,False,-1.577068,False,False,False
2,"Lackey, Andrew",False,0.151504,False,-1.722854,True,True,True
3,"Milam, Blaine Keith",False,0.242632,False,-1.138303,True,True,True
4,"Kennedy, Carlos",False,0.299126,False,-0.851463,False,False,False
...,...,...,...,...,...,...,...,...
152,Tina Brown,True,0.889831,True,2.089015,True,True,True
153,Valerie Martin,True,0.970570,True,3.495876,True,True,True
154,Veronica Gonzales,True,0.916152,True,2.391179,True,True,True
155,Virginia Caudill,True,0.925623,False,2.521317,True,True,True


In [ ]:
check_calipers_same_match(multi, 'psm_multi_cal_rec','psm_muti_cal_loose')

True

In [ ]:
multi = add_propensity(race_cols, 'gender_woman', ohe_meta, multi, 'multi_cal_rec_hm2', cal=0.338364839705966, h_m=2, add_score_col=False )


In [ ]:
multi[multi['psm_multi_cal_rec']==True].def_woman.value_counts()

,count
def_woman,
False,46
True,46


In [ ]:
check_calipers_same_match(multi, 'psm_multi_cal_rec','psm_multi_cal_rec_hm2')

False

In [ ]:
multi[multi['psm_multi_cal_rec_hm2']==True].def_woman.value_counts()

,count
def_woman,
False,92
True,46


In [ ]:
multi = add_propensity(race_cols, 'gender_woman', ohe_meta, multi, 'multi_cal_rec_rep', cal=0.338364839705966, rep=True, h_m=2, add_score_col=False )


In [ ]:
multi[multi['psm_multi_cal_rec_rep']==True].def_woman.value_counts()

,count
def_woman,
False,92
True,46


In [ ]:
check_calipers_same_match(multi, 'psm_multi_cal_rec_rep','psm_multi_cal_rec_hm2')

True